In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
from cols_to_keep import *

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

reco_cols_to_keep =  min_reco_cols_to_keep + [
    ('truth','nu_categ', '', '', '',''),
    ('truth','genie_categ', '', '', '',''),
    ('truth','nu_categ_proton_reduced', '', '', '','')
]

mc_bnb_evt_df[reco_cols_to_keep]

#Load data
keys2load = ["cc1pi_good", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight_quality_cut.df", keys2load, 100)
data_evt_df = data_df['cc1pi_good']
data_hdr_df = data_df['hdr']
del data_df
gc.collect()

keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
#load off beam light df
off_beam_light_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_offbeamlight.df", keys2load, 100)
off_beam_light_evt_df = off_beam_light_df['cc1pi']
off_beam_light_hdr_df = off_beam_light_df['hdr']
del off_beam_light_df
gc.collect()

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

# BNB data
print("data_tot_pot: %.3e" %(data_tot_pot))
print("data tot gates : %.3e" %(data_gates))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

off_beam_data_gates = off_beam_light_hdr_df.noffbeambnb.sum()
print("intime cosmics data gates: {:.2e}".format(off_beam_data_gates))
f = 0.075
scale_off_beam_to_lightdata = (1-f)*data_gates/off_beam_data_gates
print("goal scale: {:.2f}".format(scale_off_beam_to_lightdata))
off_beam_light_evt_df[pot_weight_col] = scale_off_beam_to_lightdata * np.ones(len(off_beam_light_evt_df))



In [ ]:
#Add MC stat:
mc_evt_df = mc_bnb_evt_df

# Test background composition

In [ ]:
mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
data_cumulative_masks = build_event_cumulative_masks(data_evt_df, sideband = "")

In [ ]:
for name in mc_cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   mc_cumulative_masks[name],   use_weight=True)
    n_data = get_n_evt(data_evt_df, data_cumulative_masks[name], use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")

In [ ]:
for key in ["0p", "1p", "2plusp"]:
    mask = data_cumulative_masks["energy"] & mask_dict[key](data_evt_df)
    print(f"{key}: {get_n_evt(data_evt_df, use_weight=False, mask=mask)}")

# Version that imports the systematics for the final vars

In [ ]:
def filter_df(df, mask_key, cut_mask, config):
    """Applies cut masks, extra masks, and slice grouping."""
    filtered = df[cut_mask & mask_dict[config.extra_mask](df)]
    if config.first_per_slice:
        filtered = filtered.groupby(level=SLICE_LEVELS, sort=False).first()
    return filtered

In [ ]:
final_var_configs = [config_all_evts_final, config_p_mu_final, config_cos_theta_mu_final, config_p_pi_final,
                     config_cos_theta_pi_final, config_delta_alpha_T_final, config_delta_pT_final,
                    config_delta_phi_T_final, config_num_protons,config_angle_between_candidates_final]

In [ ]:

def add_approval_text(approval, textloc_x, textloc_y, textloc_ha, fontsize = 16, ax=None):
    if approval == "internal":
        approval_text = r"$\mathbf{SBND}$ Internal"
        textcolor = 'rosybrown'
    elif approval == "preliminary":
        approval_text = r"$\mathbf{SBND}$ Preliminary"
        textcolor = 'gray'
    elif approval == "AnalysisInProgress":
        approval_text = r"$\mathbf{SBND}$ Analysis in Progress"
        textcolor = 'gray'
    else:
        return # don't add anything

    if ax is None:
        ax = plt.gca()  
    ax.text(
        textloc_x, textloc_y, 
        approval_text, 
        transform=ax.transAxes, 
        ha=textloc_ha, va='bottom',  # Changed to 'bottom' so we stack upwards cleanly
        fontsize=fontsize, color=textcolor # Fixed: now using the color assigned by the status
    )


def add_genie_version_text(textloc_x, textloc_y, textloc_ha, fontsize = 13.5, ax=None):
    if ax is None:
        ax = plt.gcf().axes[0]  
    ax.text(textloc_x, textloc_y, 
            r"GENIE v3.4.0 AR23_00i_00_000", 
            transform=ax.transAxes, 
            ha=textloc_ha, va='bottom', # Changed to 'bottom'
            fontsize=fontsize, color='gray')


def add_exposure_text(textloc_x, textloc_y, textloc_ha, fontsize = 13.5, ax=None, data_pot =1e20):
    if ax is None:
        ax = plt.gcf().axes[0]  
    ax.text(textloc_x, textloc_y, 
            f"BNB Exposure: {data_pot:.2e} POT", 
            transform=ax.transAxes, 
            ha=textloc_ha, va='bottom', # Changed to 'bottom'
            fontsize=fontsize, color='gray')


def plot_stacked_histogram_with_ratio(
    mc_df,
    data_df,
    config,
    cov_frac_matrix=None,
    cov_matrix=None,
    title: str = None,
    weight_column: tuple = None,
    data_pot: float = None,
    normalize: bool = False,
    show_stats: bool = True,
    symmetric_ratio: bool = False,
    divide_by_bin_width: bool = False,
    skip_bin_compression: bool = False
):
    # ==========================================
    # 1. PRE-PROCESSING AND SCALING
    # ==========================================
    slice_levels = ['__ntuple', 'entry', 'rec.slc..index']
    if config.first_per_slice:
        mc_df   = mc_df.groupby(level=slice_levels, sort=False).first()
        data_df = data_df.groupby(level=slice_levels, sort=False).first()

    mc_data    = mc_df[config.var_evt_reco_col]
    mc_types   = mc_df[config.truth_column]
    mc_weights = mc_df[weight_column] if weight_column in mc_df.columns else pd.Series(1.0, index=mc_df.index)

    # --- Palette Selection ---
    present_categories = set(mc_types.dropna().unique())
    palettes = [category_colors, category_colors_pfp, proton_distinction_category_colors, genie_category_colors]
    chosen_map, max_overlap = category_colors, -1
    for p in palettes:
        overlap = len(present_categories.intersection(p.keys()))
        if overlap > max_overlap:
            max_overlap = overlap
            chosen_map  = p

    # ==========================================
    # 2. SORTING CATEGORIES
    # ==========================================
    category_totals = [(t, mc_weights[mc_types == t].sum()) for t in mc_types.dropna().unique()]
    signal_keys     = ["CC1pi"]
    if config.truth_column == ('truth','genie_categ','','','',''):
        signal_keys = ["nu_mu_CC_Res"]
        
    signals     = sorted([x for x in category_totals if     x[0] in signal_keys], key=lambda x: x[1], reverse=True)
    backgrounds = sorted([x for x in category_totals if not x[0] in signal_keys], key=lambda x: x[1], reverse=True)

    sorted_types   = [x[0] for x in backgrounds + signals]
    stack_data_mc  = [mc_data[mc_types == t].dropna() for t in sorted_types]
    stack_weights  = [mc_weights[mc_types == t].loc[mc_data[mc_types == t].dropna().index] for t in sorted_types]

    # --- TRUE PHYSICS BIN TRACKING ---
    true_bins = np.array(config.bins).astype(float)
    max_bin_edge = true_bins[-1]
    bin_widths = np.diff(true_bins)
    total_true_width = true_bins[-1] - true_bins[0]

    # --- EXCEPTION 1: Check for 'all_evts' ---
    var_identifier   = config.file_name
    skip_compression = (var_identifier == "all_evts") or (len(config.bins) <= 2) or skip_bin_compression

    # --- AUTO-SCALING VISUAL BINS (MAX 50% CONSTRAINT) ---
    visual_bins = [true_bins[0]]
    compressed_indices = [] 
    
    if skip_compression:
        visual_bins = true_bins.copy()
    else:
        is_bloated = bin_widths >= (0.50 * total_true_width)
        num_bloated = np.sum(is_bloated)
        
        bloated_budget_fraction = 0.50 * num_bloated
        normal_budget_fraction = 1.0 - bloated_budget_fraction
        
        sum_normal_true_widths = np.sum(bin_widths[~is_bloated])
        if sum_normal_true_widths == 0: 
            sum_normal_true_widths = 1.0

        current_visual_coord = true_bins[0]
        for i in range(len(bin_widths)):
            if is_bloated[i]:
                visual_width = 0.50 * total_true_width
                compressed_indices.append(i)
            else:
                fraction_of_normal = bin_widths[i] / sum_normal_true_widths
                visual_width = fraction_of_normal * (normal_budget_fraction * total_true_width)
                
            current_visual_coord += visual_width
            visual_bins.append(current_visual_coord)
        
    visual_bins = np.array(visual_bins)
    visual_bin_centers = (visual_bins[:-1] + visual_bins[1:]) / 2
    visual_bin_widths  = np.diff(visual_bins)

    def map_to_visual_coordinates(data_series):
        if skip_compression:
            return np.clip(data_series, true_bins[0], max_bin_edge) if config.clip else data_series
            
        if not config.clip:
            clipped = data_series
        else:
            clipped = np.clip(data_series, true_bins[0], max_bin_edge)
        
        indices = np.digitize(clipped, true_bins) - 1
        indices = np.clip(indices, 0, len(true_bins) - 2)
        
        fractional_pos = (clipped - true_bins[indices]) / bin_widths[indices]
        visual_coords = visual_bins[indices] + fractional_pos * visual_bin_widths[indices]
        return visual_coords

    stack_mc_visual = [map_to_visual_coordinates(d) for d in stack_data_mc]

    # ==========================================
    # 3. SETUP FIGURE
    # ==========================================
    fig      = plt.figure(figsize=(10, 8))
    gs       = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.07)
    ax_top   = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_top)

    colors         = [chosen_map.get(t, "#7f7f7f") for t in sorted_types]
    all_mc_weights = pd.concat(stack_weights)

    mc_sum, _ = np.histogram(pd.concat(stack_mc_visual), bins=visual_bins, weights=all_mc_weights)

    # --- MC UNCERTAINTY ---
    if cov_frac_matrix is not None:
        frac_err = np.sqrt(np.diag(cov_frac_matrix))
        mc_error = frac_err * mc_sum
    else:
        mc_sum_w2, _ = np.histogram(pd.concat(stack_mc_visual), bins=visual_bins, weights=all_mc_weights ** 2)
        mc_error         = np.sqrt(mc_sum_w2)
        stat_cov         = get_stat_covariance_matrix(mc_sum, mc_sum_w2)
        cov_matrix       = stat_cov["cov"]
        cov_frac_matrix  = stat_cov["cov_frac"]

    if normalize:
        norm_fact = mc_sum.sum()
        if norm_fact > 0:
            mc_sum    /= norm_fact
            mc_error  /= norm_fact
            stack_weights = [w / norm_fact for w in stack_weights]

    # --- DATA ---
    cols_to_check = [config.var_evt_reco_col]
    if weight_column in data_df.columns:
        cols_to_check.append(weight_column)
    valid_df     = data_df.dropna(subset=cols_to_check)
    data         = valid_df[config.var_evt_reco_col]
    data_weights = valid_df[weight_column] if weight_column in valid_df.columns else np.ones(len(data))

    data_visual = map_to_visual_coordinates(data)
    data_counts, _ = np.histogram(data_visual, bins=visual_bins, weights=data_weights)
    data_sum_w2, _ = np.histogram(data_visual, bins=visual_bins, weights=data_weights ** 2)

    if normalize:
        denom            = len(data) if len(data) > 0 else 1
        data_plot_counts = data_counts / denom
        data_errors      = np.sqrt(data_sum_w2) / denom
    else:
        data_plot_counts = data_counts.astype(float)
        data_errors      = np.sqrt(data_sum_w2)

    # ==========================================
    # 4. STATISTICS & PLOTTING
    # ==========================================
    ret_stats_data_rate = get_stat_covariance_matrix(data_plot_counts, data_sum_w2)
    chi2_val, ndof, p_val = get_chi2(data_counts, mc_sum, cov_matrix + ret_stats_data_rate["cov"])

    if divide_by_bin_width:
        mc_sum           /= bin_widths
        mc_error         /= bin_widths
        data_plot_counts /= bin_widths
        data_errors      /= bin_widths
        
        new_stack_weights = []
        for i, d in enumerate(stack_data_mc):
            clipped_d = np.clip(d, true_bins[0], max_bin_edge) if config.clip else d
            bin_indices = np.clip(np.digitize(clipped_d, true_bins) - 1, 0, len(bin_widths) - 1)
            new_stack_weights.append(stack_weights[i] / bin_widths[bin_indices])
        stack_weights = new_stack_weights

    ax_top.hist(stack_mc_visual, bins=visual_bins, stacked=True, weights=stack_weights,
                histtype='stepfilled', color=colors, alpha=0.3)
    ax_top.hist(stack_mc_visual, bins=visual_bins, stacked=True, weights=stack_weights,
                histtype='step', color=colors, linewidth=2)
    
    ax_top.bar(visual_bin_centers, 2 * mc_error, bottom=mc_sum - mc_error, width=visual_bin_widths,
                edgecolor='grey', facecolor='grey', alpha=0.2, linewidth=0)
    ax_top.bar(visual_bin_centers, 2 * mc_error, bottom=mc_sum - mc_error, width=visual_bin_widths,
                edgecolor='grey', facecolor='none', hatch='////', alpha=0.5, linewidth=0)
    ax_top.errorbar(visual_bin_centers, data_plot_counts, yerr=data_errors, xerr=visual_bin_widths / 2,
                    fmt='ko', markersize=6, zorder=10, capsize=0)

    # ==========================================
    # 5. RATIO SUBPLOT
    # ==========================================
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio       = np.divide(data_plot_counts, mc_sum, out=np.zeros_like(data_plot_counts), where=mc_sum != 0)
        ratio_error = np.divide(data_errors,       mc_sum, out=np.zeros_like(data_errors),       where=mc_sum != 0)
        mc_rel_error = np.divide(mc_error,          mc_sum, out=np.zeros_like(mc_error),           where=mc_sum != 0)

    valid_ratio = mc_sum > 0
    if np.any(valid_ratio) and symmetric_ratio:
        data_extrema  = np.maximum(
            np.abs((ratio[valid_ratio] + ratio_error[valid_ratio]) - 1),
            np.abs((ratio[valid_ratio] - ratio_error[valid_ratio]) - 1)
        )
        mc_extrema    = mc_rel_error[valid_ratio]
        max_deviation = np.max(np.maximum(data_extrema, mc_extrema))
        y_padding     = max(max_deviation * 1.4, 0.1)
        ax_ratio.set_ylim(1 - y_padding, 1 + y_padding)
    elif np.any(valid_ratio):
        max_deviation = np.max(np.abs(ratio[valid_ratio] - 1) + ratio_error[valid_ratio])
        y_padding     = max(min(max_deviation * 1.4, 0.75), 0.1)
        ax_ratio.set_ylim(1 - y_padding, 1 + y_padding)
    else:
        ax_ratio.set_ylim(0.5, 1.5)

    ax_ratio.bar(visual_bin_centers, 2 * mc_rel_error, bottom=1 - mc_rel_error, width=visual_bin_widths,
                 edgecolor='grey', facecolor='grey', alpha=0.2, linewidth=0)
    ax_ratio.bar(visual_bin_centers, 2 * mc_rel_error, bottom=1 - mc_rel_error, width=visual_bin_widths,
                 edgecolor='grey', facecolor='none', hatch='////', alpha=0.4, linewidth=0)
    ax_ratio.errorbar(visual_bin_centers, ratio, yerr=ratio_error, xerr=visual_bin_widths / 2,
                      fmt='ko', markersize=6, capsize=0)
    ax_ratio.axhline(1.0, color='#d62728', linestyle='--', linewidth=2)

    # --- VERTICAL CUT LINE ---
    cut_val  = getattr(config, 'cut_value', None)
    draw_cut = False
    if cut_val is not None:
        draw_cut = (
            any(v != -999 for v in cut_val)
            if isinstance(cut_val, (list, np.ndarray))
            else (cut_val != -999)
        )

    cut_hand = Line2D([0], [0], color='black', linestyle='--', linewidth=2, label='Selection Cut')
    if draw_cut:
        cuts_to_draw = cut_val if isinstance(cut_val, (list, np.ndarray)) else [cut_val]
        for ax in [ax_top, ax_ratio]:
            for val in cuts_to_draw:
                if val != -999:
                    if skip_compression:
                        v_cut = val
                    else:
                        indices = np.digitize(val, true_bins) - 1
                        indices = np.clip(indices, 0, len(true_bins) - 2)
                        fractional_pos = (val - true_bins[indices]) / bin_widths[indices]
                        v_cut = visual_bins[indices] + fractional_pos * visual_bin_widths[indices]
                    ax.axvline(v_cut, color='black', linestyle='--', linewidth=2, zorder=2)

    # ==========================================
    # 6. LEGEND ENGINE DECLARATION
    # ==========================================
    total_data_counts = len(data_clipped) if 'data_clipped' in locals() else len(data)
    mc_hand  = [Patch(facecolor=to_rgba(chosen_map.get(t, "#7f7f7f"), 0.3),
                      edgecolor=chosen_map.get(t, "#7f7f7f"),
                      label=bkg_name_nice_map.get(t, t)) for t in sorted_types]
    err_label = 'Prelim. Total Unc.' if show_stats else 'MC Stat. Error'
    err_hand  = Patch(edgecolor='grey', facecolor='none', hatch='////', alpha=0.5, label=err_label)
    dat_hand  = Line2D([0], [0], color='black', marker='o', linestyle='', label='Data', markersize=8)

    all_handles = mc_hand + [err_hand, dat_hand]
    all_labels  = [h.get_label() for h in mc_hand] + [err_label, 'Data']

    if draw_cut:
        all_handles.append(cut_hand)
        all_labels.append(cut_hand.get_label())

    if show_stats:
        if ndof > 0 and chi2_val is not None:
            chi2_str = f"$\\chi^{{2}}$ / ndf = {chi2_val/ndof:.3f}"
        else:
            chi2_str = f"$\\chi^{{2}}$: N/A"
        p_value_str      = f"$p_{{\\mathrm{{value}}}}$ = {p_val:.3f}" if p_val is not None else "$p$: N/A"
        data_str         = f'$N_{{\\mathrm{{Data}}}}$ = {total_data_counts}'
        chi2_handle      = Patch(color='none', label=chi2_str)
        p_value_handle   = Patch(color='none', label=p_value_str)
        N_data_evts_handle = Patch(color='none', label=data_str)
        all_handles += [chi2_handle, p_value_handle, N_data_evts_handle]
        all_labels  += [chi2_str, p_value_str, data_str]

    n_cols = (len(all_handles) + 3) // 4
    leg = ax_top.legend(
        handles=all_handles, labels=all_labels,
        loc='upper ' + config.stats_horizontal_alignment, ncol=n_cols,
        fontsize=12, framealpha=1.0, edgecolor='black', fancybox=False,
        borderaxespad=1, columnspacing=1.5, handlelength=1.5, handletextpad=0.5,
    )
    leg.get_frame().set_linewidth(1.5)

    # ==========================================
    # 7. FINAL STYLING & TICK LABEL OVERRIDES
    # ==========================================
    ax_top.set_xlim(visual_bins[0], visual_bins[-1])
    ax_top.set_ylim(0, ax_top.get_ylim()[1] * 1.4)
    
    if var_identifier == "all_evts":
        ax_top.set_ylim(0, ax_top.get_ylim()[1] * 1.25)

     # --- DYNAMIC TICK CONFIGURATIONS ---
    if var_identifier == "num_protons_two":
        tick_positions = visual_bin_centers
        tick_labels = [f"{int(val)}" for val in true_bins[:-1]]
        if len(tick_labels) > 0:
            tick_labels[-1] = r"N"
        ax_ratio.set_xticks(tick_positions)
        ax_ratio.set_xticklabels(tick_labels, fontsize=14)
    elif var_identifier == "num_protons":
        tick_positions = visual_bin_centers
        tick_labels = [f"{int(val)}" for val in true_bins[:-1]]
        if len(tick_labels) > 0:
            tick_labels[-1] = r"$\geq$" + f"{int(true_bins[-2])}"
        ax_ratio.set_xticks(tick_positions)
        ax_ratio.set_xticklabels(tick_labels, fontsize=14)
    elif not skip_bin_compression:
        tick_positions = visual_bins
        tick_labels = [f"{val:.2f}".rstrip('0').rstrip('.') for val in true_bins]
        ax_ratio.set_xticks(tick_positions)
        ax_ratio.set_xticklabels(tick_labels, fontsize=14)
        
    ylabel = config.ylabel
    if divide_by_bin_width & (var_identifier != "all_evts") & (var_identifier != "num_protons"):
        ylabel += " / Bin width"

    ax_top.set_ylabel(f"{ylabel}")
    ax_ratio.set_ylabel("Data/MC")
    ax_ratio.set_xlabel(config.xlabel, fontsize=20)
    ax_ratio.tick_params(axis='x', which='both', direction='inout', length=6)

    # Optional: Optional custom title handling (uncomment if required)
    # if title is not None:
    #     ax_top.set_title(title, fontsize=18, pad=15)

    plt.setp(ax_top.get_xticklabels(), visible=False)
    fig.align_ylabels([ax_top, ax_ratio])
    
    # Finalize visual borders before extraction 
    plt.subplots_adjust(top=0.92, bottom=0.12, left=0.12, right=0.95, hspace=0.07)

    # ==========================================
    # 8. GEOMETRY-LOCKED RENDERING (TEXT & AXIS BREAKS)
    # ==========================================
    # Freeze canvas dimensions to accurately map pixels to relative axis scale
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    
    # --- A. Axis Break Slashes ---
    if not skip_compression and len(compressed_indices) > 0:
        for bin_idx in compressed_indices:
            break_x_data = visual_bin_centers[bin_idx]
            for ax in [ax_top, ax_ratio]:
                display_x, _ = ax.transData.transform((break_x_data, ax.get_ylim()[0]))
                x_fig = fig.transFigure.inverted().transform((display_x, 0))[0]
                display_y = ax.transAxes.transform((0, 0))[1]
                y_fig = fig.transFigure.inverted().transform((0, display_y))[1]
                
                slash_dx = 0.008
                slash_dy = 0.012
                for offset in [-0.003, 0.003]:  
                    l = Line2D(
                        [x_fig + offset - slash_dx, x_fig + offset + slash_dx],
                        [y_fig - slash_dy,           y_fig + slash_dy],
                        color='black', linewidth=1.5,
                        transform=fig.transFigure,
                        clip_on=False,
                        zorder=30
                    )
                    fig.add_artist(l)

    # --- B. Compute Legend Dimensions ---
    leg_bbox = leg.get_window_extent(renderer)
    inv_axes = ax_top.transAxes.inverted()
    
    is_left = config.stats_horizontal_alignment.lower() == "left"
    pixel_x = leg_bbox.x0 if is_left else leg_bbox.x1
    ha_style = 'left' if is_left else 'right'
    
    bottom_corner_in_axes = inv_axes.transform((pixel_x, leg_bbox.y0))
    leg_bottom_x = bottom_corner_in_axes[0]
    leg_bottom_y = bottom_corner_in_axes[1]

    # --- C. Add Watermark Texts anchored to legend in axes coords ---
    plt.draw()
    renderer  = fig.canvas.get_renderer()
    leg_bbox  = leg.get_window_extent(renderer)
    inv_axes  = ax_top.transAxes.inverted()
    
    is_left  = config.stats_horizontal_alignment.lower() == "left"
    ha_style = 'left' if is_left else 'right'
    
    # Convert both x and y of legend bottom corner to axes fraction
    leg_x_ax, leg_y_ax = inv_axes.transform(
        (leg_bbox.x0 if is_left else leg_bbox.x1, leg_bbox.y0)
    )
    
    # Convert the fixed pixel offsets (-0.06, -0.10, -0.14) to axes fraction
    # by measuring one axes unit in pixels
    axes_height_px = ax_top.get_window_extent(renderer).height
    offset_per_unit = 1.0 / axes_height_px  # 1 pixel in axes fraction
    
    # Use a fixed point size offset instead of hardcoded axes fractions
    # 14pt font ≈ 19px at 100dpi; use 1.5× line spacing
    line_height_ax = (14 * 1.5) * offset_per_unit
    
    for i, (fn, fs) in enumerate([
        (add_approval_text,     14  ),
        (add_genie_version_text, 11.5),
        (add_exposure_text,      11.5),
    ]):
        kwargs = dict(
            textloc_x=leg_x_ax,
            textloc_y=leg_y_ax - (i + 1) * line_height_ax * 1.2,
            textloc_ha=ha_style,
            ax=ax_top,
            fontsize=fs,
        )
        if fn == add_exposure_text:
            kwargs['data_pot'] = data_pot
        if fn == add_approval_text:
            kwargs['approval'] = "AnalysisInProgress"
        fn(**kwargs)

    # --- D. Shift Statistics Strings ---
    if show_stats:
        texts = leg.get_texts()
        for t in texts[-3:]:
            t.set_position((-28, 0))

    # Safe display
    plt.show()
    return fig, p_val


In [ ]:
import os, shutil, tarfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


config_p_pi_final_p_type = FullHistogramConfig(
    file_name = "pion_p_p_type",      
    var_evt_reco_col=('slc','measure_var','TLE_p_pi','','',''),
    truth_column = pion_p_type_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins= np.array([0.13, 0.218, 0.296,0.415,2]),
    xlabel=r"$P_\pi$ [GeV]",
    ylabel=slices_y_label
)


# --- Configuration & Setup ---
config_vec =  [config_p_pi_final_p_type] + final_var_configs
FULL_SYST = True

if FULL_SYST:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/FinalMCDataCompGraphsFullErr"
else:
    parent_path = "/exp/sbnd/data/users/lpelegri/Graphs/FinalMCDataCompGraphs"

'''
if plot_sideband:
    parent_path += "sideband"
'''

good_base_path, bad_base_path = os.path.join(parent_path, "good"), os.path.join(parent_path, "bad")
tar_output_path = f"{parent_path}.tar"

CHI2_THRESHOLD = 5
PLOT_IND_UNCR = True
WGT_COL = ('slc', 'wgt', '', '', '', '')
SLICE_LEVELS = ['__ntuple', 'entry', 'rec.slc..index']

# Color codes
RED, RESET = "\033[31m", "\033[0m"

# Clean workspace
for path in [parent_path, tar_output_path]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
os.makedirs(good_base_path, exist_ok=True); os.makedirs(bad_base_path, exist_ok=True)

file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices_rate_no_bkg_substract"

# Load Systematic Dictionaries
flux_syst = np.load(file_dir + "/extended_flux_syst_dict_rate_ar23p.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict_rate_ar23p.npz")
genie_syst = np.load(file_dir + "/extended_genie_syst_dict_xsec_ar23p.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict_ar23p.npz")


file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")

'''
flux_syst = np.load(file_dir + "/extended_flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/extended_g4_syst_dict.npz")
genie_syst = np.load(file_dir + "/extended_genie_xsec_syst_dict.npz")
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")
'''
pot_frac_unc = 0.02
ntargets_frac_unc = 0.01


GENIE_TAGS = [False, True]

# --- Main Analysis Loop ---
for PLOT_GENIE_CATEG in GENIE_TAGS:
    for config in config_vec:
        # Determine cut range
        idx_range = sorted([cuts.index(config.start_cut), cuts.index(config.end_cut)])
        selected_cuts = cuts[idx_range[0] : idx_range[1] + 1]
    
        if config.file_name != "pion_p_p_type":
            if PLOT_GENIE_CATEG:
                config.truth_column = ('truth','genie_categ','','','','')
            else:
                config.truth_column = ('truth','nu_categ','','','','')
            
        for cut in selected_cuts: 
            print(f"--- Processing Cut: {cut} ---")
            
            # 1. Prepare Dataframes
            curr_mc = filter_df(mc_evt_df, config.extra_mask, mc_cumulative_masks[cut], config)
            curr_data = filter_df(data_evt_df, config.extra_mask, data_cumulative_masks[cut], config)
            
            # Determine number of bins from config to initialize matrices
            # (This avoids the IndexError by matching the dimension of the loaded matrices)
            n_bins = len(config.bins) - 1
            bin_centers = (config.bins[:-1] + config.bins[1:]) / 2.
    
            # 2. Systematic Matrix Aggregation
            # We start with MC Statistics as the baseline matrix
            
            if config.file_name == "pion_p_p_type":
                total_cov_frac = mcstat_syst["pion_p"].copy()
            else:
                total_cov_frac = mcstat_syst[config.file_name].copy()
            frac_unc_list = [(np.sqrt(np.diag(total_cov_frac)), "MCStat")]
            
            if FULL_SYST:
                # Binned Systematics (Matrices)
                systs = [flux_syst, g4_syst, genie_syst, detvar_syst, cosmics_syst]
                syst_names = ["Flux", "G4", "Genie", "Detector", "Cosmic"]
    
                for name, syst_dict in zip(syst_names, systs):
                    if config.file_name == "pion_p_p_type":
                        matrix = syst_dict["pion_p"]
                    else:
                        matrix = syst_dict[config.file_name]
                        
                    total_cov_frac += matrix # Matrix addition preserves correlations
                    frac_unc_list.append((np.sqrt(np.diag(matrix)), name))
    
                # Flat Systematics (Normalization)f
                # These are 100% correlated across all bins
                flat_systs = [pot_frac_unc, ntargets_frac_unc]
                flat_names = ["POT", "Ntargets"]
    
                for name, val in zip(flat_names, flat_systs):
                    # Create a matrix where every element is (sigma_flat)^2
                    flat_matrix = np.full((n_bins, n_bins), val**2)
                    total_cov_frac += flat_matrix
                    frac_unc_list.append((val * np.ones(n_bins), name))
    
            # 3. Final Uncertainty Vector for Plotting
            total_uncertainty_vec = np.sqrt(np.diag(total_cov_frac))
            final_plot_list = [(total_uncertainty_vec, "Total")] + frac_unc_list
            #plot_frac_unc(final_plot_list, config)
            
            # 4. Convert Fractional Covariance to Absolute Covariance for Chi2
            # Use the MC counts for the current cut to scale the matrix
            # 'ret' logic usually comes from the plotter; ensure you have mc_counts here
            mc_counts, _ = np.histogram(curr_mc[config.var_evt_reco_col], bins=config.bins, weights=curr_mc[WGT_COL])
            
            total_cov = cov_from_fraccov(total_cov_frac, mc_counts)
    
            # 5. Plotting and Chi2 Calculation
            fig, red_chi2 = plot_stacked_histogram_with_ratio(
                mc_df=curr_mc, data_df=curr_data, config=config,
                cov_frac_matrix=total_cov_frac, cov_matrix=total_cov,
                title=f"{cut} - {config.file_name}", weight_column=WGT_COL,
                data_pot=data_tot_pot, normalize=False, show_stats=FULL_SYST, symmetric_ratio = True, divide_by_bin_width = True
            )
    
            # 6. File Management
            status_path = good_base_path if red_chi2 < CHI2_THRESHOLD else bad_base_path
            save_dir = os.path.join(status_path, config.file_name)
            os.makedirs(save_dir, exist_ok=True)

            topology_name = "topology"
            if PLOT_GENIE_CATEG:
                topology_name = "genie"
            f_name = f"cut_{cut}_{config.file_name}_{topology_name}.pdf"
            if config.file_name == "pion_p_p_type":
                f_name = f"cut_{cut}_{config.file_name}.pdf"
            
            fig.savefig(os.path.join(save_dir, f_name), format='pdf', bbox_inches='tight')
            plt.close(fig)
    
            color = RED if red_chi2 > CHI2_THRESHOLD else ""
            print(f"{color}Saved {f_name} (Chi2: {red_chi2:.2f}){RESET}")

# --- Archive Results ---
print(f"Archiving to {tar_output_path}...")
with tarfile.open(tar_output_path, "w:gz") as tar:
    tar.add(parent_path, arcname=os.path.basename(parent_path))
print("Done!")